# Sentiment Analysis — Model Training (Google Colab)

This notebook mirrors `model/train_model.py` from the project so you can train/explore in Colab, then download the resulting `.pkl` files into your local `model/` folder before running the FastAPI app.

**Workflow:** Data Cleaning → Text Preprocessing (NLTK) → TF-IDF → Train Model → Evaluate → Save (.pkl)

## 1. Install & import dependencies

In [ ]:
!pip install -q scikit-learn pandas numpy nltk

import re
import string
import pickle

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

## 2. Upload the dataset

Upload `sentiment_dataset.csv` (from the project's `data/` folder) using the file browser on the left, or run the cell below to use Colab's upload widget.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose sentiment_dataset.csv
DATA_PATH = list(uploaded.keys())[0]

## 3. Data cleaning + preprocessing (NLTK)

In [ ]:
STOPWORDS = set(stopwords.words('english'))
NEGATIONS = {'not', 'no', 'nor', 'never', "n't"}
STOPWORDS -= NEGATIONS  # keep negations, they flip sentiment
LEMMATIZER = WordNetLemmatizer()

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r"[^a-z\s']", ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation.replace("'", '')))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_text(text: str) -> str:
    cleaned = clean_text(text)
    tokens = word_tokenize(cleaned)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['text', 'label'])
df['clean_text'] = df['text'].apply(preprocess_text)
df = df[df['clean_text'].str.len() > 0]
print(df.shape)
df.head()

## 4. Feature engineering (TF-IDF) + train/test split

In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## 5. Train & compare candidate models

In [ ]:
candidates = {
    'LogisticRegression': LogisticRegression(max_iter=1000, C=5),
    'MultinomialNB': MultinomialNB(),
    'LinearSVC': LinearSVC(),
}

best_name, best_model, best_acc = None, None, -1
for name, model in candidates.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    print(f'{name}: accuracy = {acc:.4f}')
    if acc > best_acc:
        best_name, best_model, best_acc = name, model, acc

print(f'\nBest model: {best_name} (accuracy={best_acc:.4f})')

## 6. Evaluate the best model

In [ ]:
y_pred = best_model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred, labels=sorted(y.unique())))

## 7. Save model + vectorizer, then download

Download these two `.pkl` files and place them in your project's `model/` folder to use with the FastAPI backend.

In [ ]:
with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

from google.colab import files
files.download('sentiment_model.pkl')
files.download('tfidf_vectorizer.pkl')